# Phase 1: Project Setup and Initial Data Quality Checks

This notebook documents Phase 1 for the Kaggle Bosch Production Line Performance project.

Goals covered here:

- Confirm the project folder structure.
- Confirm the Python environment and required libraries.
- Load the train and test numeric, categorical, and date datasets.
- Confirm the target column location.
- Run initial data quality checks for dataset size, feature counts, and missing values.

## 1. Environment Setup

Run these commands once from the project root in PowerShell before using the notebook:

```powershell
python -m venv .venv
.\.venv\Scripts\python.exe -m pip install --upgrade pip
.\.venv\Scripts\python.exe -m pip install -r requirements.txt
```

The notebook uses the same dependencies listed in `requirements.txt`.

In [3]:
from pathlib import Path
import csv

import pandas as pd
from tqdm import tqdm

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 140)

print('pandas version:', pd.__version__)

pandas version: 2.2.2


## 2. Project Paths

The dataset files are currently stored in the project root. The code below also supports a future `data/raw/` location, so the raw files can be moved later without changing the analysis logic.

In [4]:
PROJECT_ROOT = Path.cwd()

# If this notebook is opened from the notebooks folder, move one level up to the project root.
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

REPORTS_DIR = PROJECT_ROOT / 'reports'
REPORTS_DIR.mkdir(exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('Reports folder:', REPORTS_DIR)

Project root: C:\Users\karin\OneDrive\Desktop\Nexturn\Bosch Production Line Performance
Reports folder: C:\Users\karin\OneDrive\Desktop\Nexturn\Bosch Production Line Performance\reports


## 3. Dataset Manifest

These are the six actual working datasets for Phase 1. The target column `Response` is expected inside `train_numeric.csv`.

The Kaggle `sample_submission.csv` file is excluded because it does not contain training information or test features.

In [5]:
DATASET_FILES = {
    'train_numeric': 'train_numeric.csv',
    'train_categorical': 'train_categorical.csv',
    'train_date': 'train_date.csv',
    'test_numeric': 'test_numeric.csv',
    'test_categorical': 'test_categorical.csv',
    'test_date': 'test_date.csv',
}

def find_dataset_path(filename: str) -> Path:
    candidates = [
        PROJECT_ROOT / filename,
        PROJECT_ROOT / 'data' / 'raw' / filename,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f'Could not find {filename} in project root or data/raw/')

dataset_paths = {name: find_dataset_path(filename) for name, filename in DATASET_FILES.items()}
dataset_paths

{'train_numeric': WindowsPath('C:/Users/karin/OneDrive/Desktop/Nexturn/Bosch Production Line Performance/train_numeric.csv'),
 'train_categorical': WindowsPath('C:/Users/karin/OneDrive/Desktop/Nexturn/Bosch Production Line Performance/train_categorical.csv'),
 'train_date': WindowsPath('C:/Users/karin/OneDrive/Desktop/Nexturn/Bosch Production Line Performance/train_date.csv'),
 'test_numeric': WindowsPath('C:/Users/karin/OneDrive/Desktop/Nexturn/Bosch Production Line Performance/test_numeric.csv'),
 'test_categorical': WindowsPath('C:/Users/karin/OneDrive/Desktop/Nexturn/Bosch Production Line Performance/test_categorical.csv'),
 'test_date': WindowsPath('C:/Users/karin/OneDrive/Desktop/Nexturn/Bosch Production Line Performance/test_date.csv')}

## 4. File Availability and Size Check

Before loading full files, check that every expected CSV exists and record its file size. This catches missing downloads or accidental file moves early.

In [6]:
file_inventory = []

for dataset, path in dataset_paths.items():
    file_inventory.append({
        'dataset': dataset,
        'filename': path.name,
        'file_size_mb': round(path.stat().st_size / (1024 ** 2), 2),
        'path': str(path),
    })

file_inventory_df = pd.DataFrame(file_inventory)
file_inventory_df

,dataset,filename,file_size_mb,path
0,train_numeric,train_numeric.csv,2040.77,C:\Users\karin\OneDrive\Desktop\Nexturn\Bosch ...
1,train_categorical,train_categorical.csv,2554.27,C:\Users\karin\OneDrive\Desktop\Nexturn\Bosch ...
2,train_date,train_date.csv,2759.33,C:\Users\karin\OneDrive\Desktop\Nexturn\Bosch ...
3,test_numeric,test_numeric.csv,2038.27,C:\Users\karin\OneDrive\Desktop\Nexturn\Bosch ...
4,test_categorical,test_categorical.csv,2554.20,C:\Users\karin\OneDrive\Desktop\Nexturn\Bosch ...
5,test_date,test_date.csv,2759.20,C:\Users\karin\OneDrive\Desktop\Nexturn\Bosch ...


## 5. Lightweight Dataset Loading

The Bosch CSV files are large, so we first load only five rows from each file. This proves the files are readable and shows the shape of each dataset without consuming too much memory.

In [7]:
samples = {}

for dataset, path in dataset_paths.items():
    samples[dataset] = pd.read_csv(path, nrows=5)
    print(f'{dataset}: {samples[dataset].shape[0]} sample rows x {samples[dataset].shape[1]} columns')

train_numeric: 5 sample rows x 970 columns
train_categorical: 5 sample rows x 2141 columns
train_date: 5 sample rows x 1157 columns
test_numeric: 5 sample rows x 969 columns
test_categorical: 5 sample rows x 2141 columns
test_date: 5 sample rows x 1157 columns


## 6. Target Column Check

For this Kaggle competition, `Response` is the target. It should be present in `train_numeric.csv`, and it should not be present in the test feature files.

In [8]:
target_check = []

for dataset, sample in samples.items():
    target_check.append({
        'dataset': dataset,
        'has_id': 'Id' in sample.columns,
        'has_response_target': 'Response' in sample.columns,
        'columns': sample.shape[1],
    })

pd.DataFrame(target_check)

,dataset,has_id,has_response_target,columns
0,train_numeric,True,True,970
1,train_categorical,True,False,2141
2,train_date,True,False,1157
3,test_numeric,True,False,969
4,test_categorical,True,False,2141
5,test_date,True,False,1157


## 7. Full Data Quality Profiling

The next function reads each large CSV in chunks. For every dataset it records:

- Row count
- Column count
- Feature count, excluding `Id` and `Response`
- Whether `Id` exists
- Whether `Response` exists
- Total missing values
- Overall missing-value percentage
- Per-column missing-value counts

Chunked reading keeps memory usage controlled while still scanning the full files.

In [9]:
def read_header(path: Path) -> list[str]:
    with path.open('r', newline='', encoding='utf-8') as file:
        return next(csv.reader(file))


def profile_csv(dataset: str, path: Path, chunksize: int = 20_000):
    columns = read_header(path)
    missing_counts = pd.Series(0, index=columns, dtype='int64')
    rows = 0

    reader = pd.read_csv(path, chunksize=chunksize, low_memory=False)
    for chunk in tqdm(reader, desc=f'Profiling {dataset}', unit='chunk'):
        rows += len(chunk)
        missing_counts = missing_counts.add(chunk.isna().sum(), fill_value=0).astype('int64')

    total_cells = rows * len(columns)
    total_missing = int(missing_counts.sum())
    id_present = 'Id' in columns
    target_present = 'Response' in columns
    feature_columns = len(columns) - int(id_present) - int(target_present)

    profile = {
        'dataset': dataset,
        'path': str(path),
        'file_size_mb': round(path.stat().st_size / (1024 ** 2), 2),
        'rows': rows,
        'columns': len(columns),
        'feature_columns': feature_columns,
        'id_present': id_present,
        'target_present': target_present,
        'total_missing_values': total_missing,
        'missing_value_pct': round((total_missing / total_cells) * 100, 4) if total_cells else 0.0,
    }

    missing_df = (
        missing_counts.rename('missing_values')
        .reset_index()
        .rename(columns={'index': 'column'})
    )
    missing_df.insert(0, 'dataset', dataset)
    missing_df['rows'] = rows
    missing_df['missing_pct'] = (missing_df['missing_values'] / rows * 100).round(4)
    return profile, missing_df

Run the full scan below. On this machine, the full Phase 1 scan can take several minutes because the six working CSVs are multi-GB files.

In [10]:
profiles = []
missing_frames = []

for dataset, path in dataset_paths.items():
    profile, missing_df = profile_csv(dataset, path, chunksize=20_000)
    profiles.append(profile)
    missing_frames.append(missing_df)

profiles_df = pd.DataFrame(profiles)
missing_by_column = pd.concat(missing_frames, ignore_index=True)

profiles_df

Profiling train_numeric: 60chunk [01:27,  1.45s/chunk]
Profiling train_categorical: 60chunk [04:11,  4.19s/chunk]
Profiling train_date: 60chunk [01:45,  1.76s/chunk]
Profiling test_numeric: 60chunk [01:30,  1.50s/chunk]
Profiling test_categorical: 60chunk [04:03,  4.07s/chunk]
Profiling test_date: 60chunk [01:55,  1.92s/chunk]


,dataset,path,file_size_mb,rows,columns,feature_columns,id_present,target_present,total_missing_values,missing_value_pct
0,train_numeric,C:\Users\karin\OneDrive\Desktop\Nexturn\Bosch ...,2040.77,1183747,970,968,True,True,929125166,80.9177
1,train_categorical,C:\Users\karin\OneDrive\Desktop\Nexturn\Bosch ...,2554.27,1183747,2141,2140,True,False,2465567643,97.2840
2,train_date,C:\Users\karin\OneDrive\Desktop\Nexturn\Bosch ...,2759.33,1183747,1157,1156,True,False,1125431152,82.1725
3,test_numeric,C:\Users\karin\OneDrive\Desktop\Nexturn\Bosch ...,2038.27,1183748,969,968,True,False,929173660,81.0054
4,test_categorical,C:\Users\karin\OneDrive\Desktop\Nexturn\Bosch ...,2554.20,1183748,2141,2140,True,False,2465604793,97.2854
5,test_date,C:\Users\karin\OneDrive\Desktop\Nexturn\Bosch ...,2759.20,1183748,1157,1156,True,False,1125501041,82.1776


## 8. Missing-Value Summary

The Bosch dataset is extremely sparse. The table below shows the ten most-missing columns per dataset, which helps identify columns that may need to be dropped or handled carefully in later phases.

In [11]:
top_missing = (
    missing_by_column.sort_values(['dataset', 'missing_values'], ascending=[True, False])
    .groupby('dataset')
    .head(10)
    .reset_index(drop=True)
)

top_missing[['dataset', 'column', 'missing_values', 'missing_pct']]

,dataset,column,missing_values,missing_pct
0,test_categorical,L0_S15_F396,1183748,100.0000
1,test_categorical,L0_S15_F399,1183748,100.0000
2,test_categorical,L0_S15_F402,1183748,100.0000
3,test_categorical,L0_S15_F405,1183748,100.0000
4,test_categorical,L0_S15_F408,1183748,100.0000
5,test_categorical,L0_S15_F411,1183748,100.0000
6,test_categorical,L0_S15_F414,1183748,100.0000
7,test_categorical,L0_S15_F417,1183748,100.0000
8,test_categorical,L0_S15_F420,1183748,100.0000
9,test_categorical,L1_S24_F1157,1183748,100.0000


## 10. Phase 1 Findings

After running the notebook, Phase 1 should confirm:

- Train files have 1,183,747 rows.
- Test files have 1,183,748 rows.
- `train_numeric.csv` contains `Id`, 968 numeric features, and the `Response` target.
- Test feature files do not include `Response`.
- The dataset is highly sparse, especially categorical and date features.

These findings shape Phase 2: memory-efficient EDA, feature selection, sparse-feature handling, and target imbalance review.